# 09 — MLP model

Build and compare the A2 deep MLP using the completed **00 → 01 → 02** workflow.

## Setup before running

**Assumption:** notebooks 00, 01 and 02 have already been run in this repository. Their outputs remain on disk; their Python variables are not needed. You do not need to rerun them for this notebook.

1. **Open the complete repository folder in VS Code**, not just this notebook. Use the existing `.venv` with Python **3.10 or 3.11**.
2. **Install this notebook's dependencies.** Open a terminal in the repository root and run the command for your system:

   macOS / Linux:
   ```bash
   .venv/bin/python -m pip install -r requirements-09.txt
   ```
   Windows PowerShell:
   ```powershell
   .\.venv\Scripts\python.exe -m pip install -r requirements-09.txt
   ```

3. **Select the kernel:** click **Select Kernel → Python Environments → the project's `.venv`**. If a kernel is already running after package installation, restart it.
4. **Check the generated inputs:**
   - `data/processed/d1_train.csv` — frozen development data from 00.
   - `data/processed/model_cv_assignments.csv` — saved grouped folds from 02. The notebook verifies these when present; otherwise it regenerates them using 02's shared code and seed.
   - `reports/tables/model_logreg_tf_idf_only_fold_scores.csv` — optional saved 02 baseline for the final comparison table.
5. **Click Run All.** The default is **5 repeats × 5 grouped folds**. Matching completed folds are reused; remaining folds train locally. Keep your laptop plugged in and awake. For a short diagnostic, change `N_REPEATS = 1` and `OUTER_LIMIT = 1` in the configuration cell.

No Colab upload, Kaggle token or NLTK resource download is required at this stage. The final test is not loaded. Metrics display inside the notebook; no metric/report files are written.

In [ ]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "src/d1_a2_mlp.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete repository folder, not a standalone notebook copy.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import d1_a2_mlp as experiment

DATA_FILE = ROOT / "data/processed/d1_train.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError("Missing data/processed/d1_train.csv from the completed notebook 00. Restore the agreed frozen file in this repository before continuing.")
train = experiment.load_development(DATA_FILE)
data_sha = hashlib.sha256(train.to_csv(index=False, lineterminator="\n").encode()).hexdigest()
RECORDED_DATA_SHA = "bb132ac17cadb39be336612523330fc85c1ccb6512ed44b6ac3ca00eb572f5e8"
print(f"Development pool: {len(train):,} postings | {int(train.fraudulent.sum()):,} frauds | {train.group_id.nunique():,} groups")
print("SHA256:", data_sha)
print("Matches recorded data:", data_sha == RECORDED_DATA_SHA)
if data_sha != RECORDED_DATA_SHA:
    print("Different frozen data: resolve the team version before claiming reproduction of the recorded scores.")
print("Reserved final test not loaded.")

## 1. Starting point → selected model

1. Keep the team's **A2 inputs and grouped folds**; use A2 logistic regression as the matched baseline.
2. Replace the classifier with a **512 → 256 → 128 → 64 → sigmoid MLP**, using ReLU, dropout **0.4 / 0.4 / 0.3 / 0.3**, and balanced class weights.
3. Evaluate the model with training-only epoch and threshold selection. The completed first five-fold run favored the MLP in AP and F1 on every fold. The character-feature addition tested in 08 did not improve the overall balance, so retain **A2 only**.

This selects a model for further development; the final-test performance is unknown.

## 2. Training choices and tuning actually performed

| Component | Current choice | How it was chosen |
|---|---|---|
| Inputs | A2, up to 30,000 word/bigram features | Carried forward from the team |
| Network and dropout | Four hidden layers above | Fixed candidate; no successful architecture search established |
| Optimizer | Adam 0.001; batch 64 | Retained setting; 0.001 is the only MLP learning-rate candidate in this grouped run |
| Class weights | Balanced, recomputed per fit | Fixed imbalance treatment |
| Training duration | Up to 50 epochs, patience 5 | Lowest loss on separate stopping groups inside training; then refit for that fixed duration |
| Classification threshold | One cutoff per outer fold | Highest F1 on training-side inner out-of-fold predictions |
| LR baseline | C = 0.1, 0.3 or 1.0 | Highest mean inner AP |

These are the actual selection steps—not a claim that the MLP learning rate or architecture was exhaustively fine-tuned. Earlier random-row tuning is not evidence of grouped performance. Model settings, checkpoint and threshold never use outer scoring labels.

In [ ]:
# Match 02: 5 repeats × 5 outer folds. Completed reference folds resume.
# To limit a diagnostic run, set N_REPEATS = 1 and OUTER_LIMIT = 1.
N_REPEATS = 5
OUTER_LIMIT = None
CONFIG = experiment.GroupedConfig(
    repeats=N_REPEATS, outer_limit=OUTER_LIMIT,
    outer_splits=5, inner_splits=3, checkpoint_splits=5,
    tfidf_max_features=30000, dense_weight=1.0,
    lr_Cs=(0.1, 0.3, 1.0), mlp_rates=(0.001,),
    max_epochs=50, patience=5, batch_size=64, seed=42)

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print("TensorFlow:", tf.__version__, "| GPUs:", gpus or "None — using CPU")

## 3. Run the grouped model comparison

Reuse 02's fold assignments when present and its shared generator otherwise. Only the MLP's additional stopping/refit step is new; details of the common evaluation remain in 02. Completed model/fold predictions resume from the existing cache. Keep the laptop awake during training.

In [ ]:
team_assignments = ROOT / "data/processed/model_cv_assignments.csv"
FOLDS = experiment.split_plan(train, CONFIG, team_assignments=team_assignments)
result = experiment.run_comparison(
    train, CONFIG, ROOT / "data/processed/09_a2_mlp", folds=FOLDS)
print(f"Completed {result['completed_outer_folds']}/{result['total_outer_folds']} configured outer folds.")

## 4. Compare metrics and tuning choices

The main comparison uses **mean outer-fold metrics**, as in 02. Combined decisions are kept separately for each repeat; the same postings are never counted as new independent examples across repeats.

Compare the **A2 deep MLP**, its matched **A2 logistic-regression baseline**, and 02's saved **TF-IDF-only logistic regression** when its exact folds match. Missing or mismatched saved scores are excluded.

In [ ]:
from src.d1_model_metrics import compare_models
metrics = compare_models(ROOT, train, CONFIG, result)
print("Mean outer-fold comparison:")
display(metrics["model_comparison"][["model", "evaluated_folds", "expected_folds", "complete",
    "tfidf_max_features", "pr_auc", "precision", "recall", "f1", "false_positive_rate"]].round(4))
print("Combined decisions per repeat:")
display(metrics["combined_decisions_by_repeat"].round(4))
print("MLP epoch, settings and threshold selected inside training:")
display(metrics["mlp_tuning_choices"].round(4))
excluded = metrics["comparison_sources"].query("status == 'excluded'")
if not excluded.empty:
    print("Historical results excluded from this comparison:")
    display(excluded[["model", "reason"]])
print("Reserved final test not accessed.")

## Result checkpoint and GitHub dependencies

The **completed one-repeat/five-fold checkpoint** had MLP mean AP/F1 **0.6776 / 0.6188**, versus A2 LR **0.5914 / 0.5151**. Combined-decision MLP F1 was **0.6286**, with **391 frauds caught / 194 false alarms**. These are five-fold results; the new configuration requests **25 folds**. Use the displayed tables after that run for the final matched development comparison.

Commit this notebook, `requirements-09.txt`, `src/d1_a2_mlp.py`, `src/d1_row_neural.py` and **`src/d1_model_metrics.py`**, alongside the existing team helpers/notebook 00. Keep data, caches, credentials and `.venv` excluded. Matching 02 text-only fold scores are an optional comparison input.